In [1]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, AutoTokenizer
from qwen_vl_utils import process_vision_info
import os

/home/jack/anaconda3/envs/qwenvl-new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
checkpoint_path = "/data3/qwen-weights/output_7b/checkpoint-829"
processor_dir = "/data3/qwen-weights/output_7b"
base_model_name = "Qwen/Qwen2.5-VL-7B-Instruct"

print("Loading fine-tuned model...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    checkpoint_path,
    torch_dtype="auto",
    device_map="auto"
)

print("Loading processor...")
processor = AutoProcessor.from_pretrained(processor_dir)

if not hasattr(processor, 'chat_template') or processor.chat_template is None:
    print("Chat template missing, loading from base model...")
    base_processor = AutoProcessor.from_pretrained(base_model_name)
    processor.chat_template = base_processor.chat_template

print("Loading tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
except:
    tokenizer = AutoTokenizer.from_pretrained(processor_dir)

print("✓ All components loaded successfully!")

Loading fine-tuned model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.57s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading processor...
Chat template missing, loading from base model...
Loading tokenizer...
✓ All components loaded successfully!


In [ ]:
# def chat_with_model(user_input, image_path=None, conversation_history=None):
#     if conversation_history is None:
#         conversation_history = []

#     if image_path and os.path.exists(image_path):
#         user_message = {
#             "role": "human",
#             "content": [
#                 {"type": "image", "image": image_path},
#                 {"type": "text", "text": user_input}
#             ]
#         }
#     else:
#         user_message = {
#             "role": "human",
#             "content": [{"type": "text", "text": user_input}]
#         }
    
#     conversation_history.append(user_message)
    
#     text = processor.apply_chat_template(
#         conversation_history, 
#         tokenize=False, 
#         add_generation_prompt=True
#     )
    
#     image_inputs, video_inputs = process_vision_info(conversation_history)
    
#     inputs = processor(
#         text=[text],
#         images=image_inputs,
#         videos=video_inputs,
#         padding=True,
#         return_tensors="pt"
#     ).to(model.device)
    
#     with torch.no_grad():
#         generated_ids = model.generate(
#             **inputs,
#             max_new_tokens=512,
#             do_sample=True,
#             temperature=0.7,
#             top_p=0.8
#         )
    
#     response_ids = [
#         out_ids[len(in_ids):] for in_ids, out_ids in 
#         zip(inputs.input_ids, generated_ids)
#     ]

#     response = processor.batch_decode(
#         response_ids, 
#         skip_special_tokens=True, 
#         clean_up_tokenization_spaces=False
#     )[0]
    
#     assistant_message = {
#         "role": "gpt",
#         "content": [{"type": "text", "text": response}]
#     }
#     conversation_history.append(assistant_message)
    
#     return response, conversation_history

# print("Chat function defined!")

Chat function defined!


In [ ]:
def chat_with_model(user_input, image_paths=None, conversation_history=None):
    if conversation_history is None:
        conversation_history = []

    content = []

    if image_paths:
        if isinstance(image_paths, str):
            image_paths = [image_paths]
        
        for image_path in image_paths:
            if os.path.exists(image_path):
                content.append({"type": "image", "image": image_path})
            else:
                print(f"Warning: Image not found: {image_path}")
    
    content.append({"type": "text", "text": user_input})
    
    user_message = {
        "role": "human",
        "content": content
    }
    
    conversation_history.append(user_message)
    
    text = processor.apply_chat_template(
        conversation_history, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    image_inputs, video_inputs = process_vision_info(conversation_history)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.8
        )
    
    response_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in 
        zip(inputs.input_ids, generated_ids)
    ]

    response = processor.batch_decode(
        response_ids, 
        skip_special_tokens=True, 
        clean_up_tokenization_spaces=False
    )[0]
    
    assistant_message = {
        "role": "gpt",
        "content": [{"type": "text", "text": response}]
    }
    conversation_history.append(assistant_message)
    
    return response, conversation_history

print("Chat function defined!")

Multi-image chat function defined!


In [ ]:
def print_conversation_history(history):
    """Print the conversation history in a readable format"""
    print("=== Conversation History ===")
    for i, message in enumerate(history):
        role = message["role"].upper()
        content = message["content"]
        print(f"{i+1}. {role}:")
        for item in content:
            if item["type"] == "text":
                print(f"   {item['text']}")
            elif item["type"] == "image":
                print(f"   [Image: {item['image']}]")
        print()

def reset_conversation():
    """Return empty conversation history"""
    return []

In [ ]:
response, history = chat_with_model("Hello! How are you today?")
print(f"Human: Hello! How are you today?")
print(f"GPT: {response}")

Human: Hello! How are you today?
GPT: "thoughts🤔" The human is greeting me. I should respond warmly and offer assistance.
"actions🚀" []
"value👉" Feel free to reach out if you need anything else.



In [ ]:
image_path = "/home/jack/Projects/yixin-llm/images_train/23251064_F5.jpg"
question = "What type of diagnostic imaging does this image represent?"

image_conversation = []

response, image_conversation = chat_with_model(
    question, 
    image_path=image_path, 
    conversation_history=image_conversation
)
print(f"Human: {question}\n")
print(f"GPT: {response}\n")


Human: What type of diagnostic imaging does this image represent?

GPT: "thoughts🤔" I need to use the BiomedClip model to ensure precise classification.
"actions🚀" [{"API_name": "BiomedClip", "API_params": {}}]
"value👉" I'll use BiomedClip to complete this request based on our current analysis.




In [6]:
question2 = "Following up on the previous analysis, what is the main focus of the image?"

response2, image_conversation = chat_with_model(
    question2, 
    image_path="/home/jack/Projects/yixin-llm/images_train/WORD_0049_0137.jpg",
    conversation_history=image_conversation
)
print(f"Human: {question2}\n")
print(f"GPT: {response2}")

Human: Following up on the previous analysis, what is the main focus of the image?

GPT: "thoughts🤔" Building on the previous BiomedClip results. Invoking LLaVA-Med for in-depth analysis.
"actions🚀" [{"API_name": "LLaVA-Med", "API_params": {"image": "/home/jack/Projects/yixin-llm/images_train/27051683_Fig2.jpg", "question": "What is the main focus of the image?"}}]
"value👉" I'll use LLaVA-Med to complete this request based on our current analysis.



In [7]:
question3 = "Now I have a second image. \nRegister this new scan with the previous one."

response3, image_conversation = chat_with_model(
    question3, 
    image_path="/home/jack/Projects/yixin-llm/yixin-llm-data/UltraSam/dataset/BrEaST/BrEaST-Lesions_USG-images_and_masks-Dec-15-2023/images/BrEaST-Lesions_USG-images_and_masks-Dec-15-2023__00224.png",
    conversation_history=image_conversation
)
print(f"Human: {question3}\n")
print(f"GPT: {response3}")

Human: Now I have a second image. 
Register this new scan with the previous one.

GPT: "thoughts🤔" Building on the previous BiomedClip, LLaVA-Med results. This is an image registration task. I need to use UniGradICON to align the moving image to the fixed image.
"actions🚀" [{"API_name": "UniGradICON", "API_params": {"modality": "CT"}}]
"value👉" I'll use UniGradICON to complete this request based on our current analysis.



In [9]:
print_conversation_history(image_conversation)

=== Conversation History ===
1. HUMAN:
   [Image: /home/jack/Projects/yixin-llm/images_train/23251064_F5.jpg]
   What type of diagnostic imaging does this image represent?

2. GPT:
   "thoughts🤔" I need to use the BiomedClip model to ensure precise classification.
"actions🚀" [{"API_name": "BiomedClip", "API_params": {}}]
"value👉" I'll use BiomedClip to complete this request based on our current analysis.


3. HUMAN:
   [Image: /home/jack/Projects/yixin-llm/images_train/WORD_0049_0137.jpg]
   Following up on the previous analysis, what is the main focus of the image?

4. GPT:
   "thoughts🤔" Building on the previous BiomedClip results. Invoking LLaVA-Med for in-depth analysis.
"actions🚀" [{"API_name": "LLaVA-Med", "API_params": {"image": "/home/jack/Projects/yixin-llm/images_train/27051683_Fig2.jpg", "question": "What is the main focus of the image?"}}]
"value👉" I'll use LLaVA-Med to complete this request based on our current analysis.


5. HUMAN:
   [Image: /home/jack/Projects/yixin-llm

In [10]:
reset_conversation()

[]

In [11]:
response, history = chat_with_model("hi")
print(f"Human: hi")
print(f"GPT: {response}")

Human: hi
GPT: "thoughts🤔" The user is greeting me. I should respond warmly and offer assistance.
"actions🚀" []
"value👉" Glad I could help!



In [17]:
image_path = ["/home/jack/Projects/yixin-llm/yixin-llm-data/instruct_dataset/mimic-cxr-5k/5k/263caa43-d7645fc2-ec153511-fefb77fe-01cf1b02.jpg", "/home/jack/.cache/kagglehub/datasets/deathtrooper/multichannel-glaucoma-benchmark-dataset/versions/10/full-fundus/full-fundus/ORIGA-531.png"]
question = "Align this for image fusion."

image_conversation = []

response, image_conversation = chat_with_model(
    question, 
    image_paths=image_path, 
    conversation_history=image_conversation
)
print(f"Human: {question}\n")
print(f"GPT: {response}\n")

Human: Align this for image fusion.

GPT: "thoughts🤔" This is an image registration task. I need to use UniGradICON to align the moving image to the fixed image.
"actions🚀" [{"API_name": "UniGradICON", "API_params": {"modality": "CT"}}]
"value👉" I'll use UniGradICON to complete this request based on our current analysis.




In [ ]:
question2 = "Could you generate a structured report for the attached MRI scan?"

response2, image_conversation = chat_with_model(
    question2, 
    conversation_history=image_conversation
)
print(f"Human: {question2}\n")
print(f"GPT: {response2}")

Human: Could you generate a structured report for the attached MRI scan?

GPT: "thoughts🤔" To provide an accurate and comprehensive structured report, I will utilize the ChatCAD-G tool to analyze the MRI scan thoroughly.
"actions🚀" [{"API_name": "ChatCAD-G", "API_params": {}}]
"value👉" I'll use ChatCAD-G to complete this request based on our current analysis.



In [19]:
question3 = "Building on the results so far, segment the target anatomy inside box [171.0, 50.0, 298.0, 193.0] for ultrasound image."

response3, image_conversation = chat_with_model(
    question3, 
    conversation_history=image_conversation
)
print(f"Human: {question3}\n")
print(f"GPT: {response3}")


Human: Building on the results so far, segment the target anatomy inside box [171.0, 50.0, 298.0, 193.0] for ultrasound image.

GPT: "thoughts🤔" This is an ultrasound segmentation task; I'll call the UltraSAM tool with a bbox prompt.
"actions🚀" [{"API_name": "UltraSAM", "API_params": {"image": 222, "prompt": {"bboxes": [[171.0, 50.0, 298.0, 193.0]]}}}]
"value👉" I'll use UltraSAM to complete this request based on our current analysis.



In [21]:
print_conversation_history(image_conversation)

=== Conversation History ===
1. HUMAN:
   [Image: /home/jack/Projects/yixin-llm/yixin-llm-data/instruct_dataset/mimic-cxr-5k/5k/263caa43-d7645fc2-ec153511-fefb77fe-01cf1b02.jpg]
   [Image: /home/jack/.cache/kagglehub/datasets/deathtrooper/multichannel-glaucoma-benchmark-dataset/versions/10/full-fundus/full-fundus/ORIGA-531.png]
   Align this for image fusion.

2. GPT:
   "thoughts🤔" This is an image registration task. I need to use UniGradICON to align the moving image to the fixed image.
"actions🚀" [{"API_name": "UniGradICON", "API_params": {"modality": "CT"}}]
"value👉" I'll use UniGradICON to complete this request based on our current analysis.


3. HUMAN:
   Could you generate a structured report for the attached MRI scan?

4. GPT:
   "thoughts🤔" To provide an accurate and comprehensive structured report, I will utilize the ChatCAD-G tool to analyze the MRI scan thoroughly.
"actions🚀" [{"API_name": "ChatCAD-G", "API_params": {}}]
"value👉" I'll use ChatCAD-G to complete this request 

In [ ]:
# Interactive chat session
print("Starting interactive chat. Type 'quit' to exit, 'reset' to clear history.")
interactive_history = []

while True:
    user_input = input("\nHuman: ")
    
    if user_input.lower() == 'quit':
        break
    elif user_input.lower() == 'reset':
        interactive_history = []
        print("Conversation history cleared.")
        continue
    
    try:
        response, interactive_history = chat_with_model(user_input, conversation_history=interactive_history)
        print(f"GPT: {response}")
    except Exception as e:
        print(f"Error: {e}")

Starting interactive chat. Type 'quit' to exit, 'reset' to clear history.
